In [2]:
# ==========================================================
#  PREDICTION & SUBMISSION NOTEBOOK
# ==========================================================
#  OBJECTIVE:
# - Load trained model
# - Generate predictions
# - Save output for evaluation

In [1]:
# ==========================================================
#  STEP 1 — LOAD LIBRARIES
# ==========================================================
import pandas as pd
import pickle

In [2]:
# ==========================================================
#  STEP 2 — LOAD TEST DATA
# ==========================================================
test_df = pd.read_csv("../data/test.csv")

#  Keep a copy of actual values (IMPORTANT for evaluation)
#  STUDENT TASK: Change target if needed

TARGET_COLUMN = "target_churn"   #  CHANGE THIS

y_true = test_df[TARGET_COLUMN]

In [3]:
# ==========================================================
#  STEP 3 — LOAD MODEL
# ==========================================================
model = pickle.load(open("../models/model.pkl", "rb"))

# ==========================================================
# STEP 4 — PREPROCESS DATA
# ==========================================================

# Fill missing values
test_df.fillna(0, inplace=True)

# Convert categorical variables
test_df = pd.get_dummies(test_df, drop_first=True)

# ==========================================================
# 🔥 STEP 4.1 — RECREATE FEATURES (VERY IMPORTANT)
# ==========================================================

test_df["spend_per_transaction"] = test_df["total_spend"] / (test_df["num_transactions"] + 1)

test_df["activity_score"] = test_df["num_transactions"] / (test_df["tenure_months"] + 1)

test_df["engagement_ratio"] = test_df["last_login_days"] / (test_df["tenure_months"] + 1)

test_df["high_spender_flag"] = (test_df["total_spend"] > test_df["total_spend"].median()).astype(int)

test_df["frequent_user_flag"] = (test_df["num_transactions"] > test_df["num_transactions"].median()).astype(int)

test_df["customer_value_score"] = (
    test_df["total_spend"] * test_df["num_transactions"]
) / (test_df["tenure_months"] + 1)

In [4]:
# ==========================================================
#  STEP 5 — ALIGN COLUMNS WITH TRAINING DATA
# ==========================================================
# Load training columns
train_cols = pd.read_csv("../data/processed_train.csv").drop(columns=[TARGET_COLUMN]).columns

# Align columns
test_df = test_df.reindex(columns=train_cols, fill_value=0)

In [5]:
# ==========================================================
#  STEP 6 — GENERATE PREDICTIONS
# ==========================================================
# predictions = model.predict(test_df)

probs = model.predict_proba(test_df)[:, 1]

predictions = (probs > 0.2).astype(int)

In [6]:
for t in [0.5, 0.3, 0.2, 0.15]:
    preds = (probs > t).astype(int)
    print(f"\nThreshold {t}")
    print(pd.Series(preds).value_counts())


Threshold 0.5
0    954
1     46
Name: count, dtype: int64

Threshold 0.3
0    934
1     66
Name: count, dtype: int64

Threshold 0.2
0    881
1    119
Name: count, dtype: int64

Threshold 0.15
0    799
1    201
Name: count, dtype: int64


In [7]:
print(pd.Series(predictions).value_counts())

0    881
1    119
Name: count, dtype: int64


In [8]:
from sklearn.metrics import classification_report

predictions = (probs > 0.2).astype(int)

print(classification_report(y_true, predictions))

              precision    recall  f1-score   support

           0       0.92      0.92      0.92       882
           1       0.38      0.38      0.38       118

    accuracy                           0.85      1000
   macro avg       0.65      0.65      0.65      1000
weighted avg       0.85      0.85      0.85      1000



In [9]:
# ==========================================================
#  STEP 7 — CREATE OUTPUT FILE
# ==========================================================
#  IMPORTANT:
# This format is REQUIRED for evaluation system

submission = pd.DataFrame({
    "actual": y_true,
    "prediction": predictions
})

submission.tail(20)

,actual,prediction
980,0,0
981,1,0
982,0,0
983,0,0
984,0,0
985,0,0
986,0,0
987,0,0
988,0,0
989,0,0


In [10]:
# ==========================================================
#  SAVE FINAL PREDICTIONS (MANDATORY STEP)
# ==========================================================

# 👉 STUDENT TASK:
# Replace YOURNAME with your actual name
# Example:
# "harshit_predictions.csv"

FILE_NAME = "Leela_predictions.csv"   # 🔥 CHANGE THIS

# 👉 Save inside outputs folder (DO NOT CHANGE PATH)
submission.to_csv(f"../outputs/{FILE_NAME}", index=False)

print("✅ File saved successfully!")

# ==========================================================
# 📌 IMPORTANT INSTRUCTIONS
# ==========================================================
# ✔ File MUST be saved in: outputs/
# ✔ File name MUST follow: YOURNAME_predictions.csv
# ✔ Columns MUST be: actual, prediction
# ✔ Do NOT change column names

# ==========================================================
# 🎯 FINAL CHECK
# ==========================================================
# Go to file explorer → outputs/
# Verify your file is present there

✅ File saved successfully!
